# 13 - Intermediate Fusion Transfer Learning (ResNet18 + FCNN - 7 Class)

**Tujuan:** Melatih model Intermediate Fusion dengan Transfer Learning yang menggabungkan fitur penampilan (ResNet18 pretrained) dan fitur geometrik (FCNN) di level fitur (feature-level) untuk pengenalan emosi.

**Catatan terminologi:** Arsitektur ini disebut **Intermediate Fusion** (Boulahia et al., 2021) karena fusi dilakukan di level representasi fitur yang diekstrak masing-masing subnetwork, bukan di level input (early fusion) maupun di level keputusan (late fusion). Istilah lain yang setara: *feature-level fusion* atau *middle fusion*. Ini **berbeda dengan hybrid fusion** yang menggabungkan dua atau lebih strategi fusi sekaligus.

**Menjawab RQ3:** *Bagaimana performa model multimodal Intermediate Fusion Transfer Learning (ResNet18 + FCNN) dalam pengenalan emosi mahasiswa pada konteks pembelajaran pemrograman menggunakan gabungan fitur penampilan dan fitur geometrik?*

**Arsitektur:**
- Image stream (ResNet18 pretrained): ResNet18 backbone → 256-dim feature vector
- Landmark stream (FCNN): 4 Dense layers → 128-dim feature vector
- Concatenate (384-dim) → Dense(512→256) → 7 classes

**3 Skenario Imbalance:**
- B1: Tanpa penanganan (baseline)
- B2: Dengan class weights (Cui et al., 2019)
- B3: Dengan class weights + augmentasi

## 1. Setup

In [ ]:
import sys
import os
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import IntermediateFusionTransfer
from training.utils import (
    EmotionMultimodalDataset, get_class_weights,
    train_model, full_evaluation,
    plot_training_history, plot_confusion_matrix, plot_per_class_f1
)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Config
DATASET_DIR = PROJECT_ROOT / "data" / "dataset"
DATASET_AUG_DIR = PROJECT_ROOT / "data" / "dataset_augmented"
OUTPUT_DIR = PROJECT_ROOT / "models" / "intermediate_fusion_transfer"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 16
EPOCHS = 80
LR = 0.00005
PATIENCE = 25
NUM_CLASSES = 7

EMOTIONS = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 2. Load Data

In [ ]:
from torch.utils.data import DataLoader
from collections import Counter

def load_dataloaders(dataset_dir, batch_size=16):
    """Load train/val/test dataloaders for multimodal (images + landmarks)."""
    train_ds = EmotionMultimodalDataset(
        dataset_dir / "X_train_images.npy",
        dataset_dir / "X_train_landmarks.npy",
        dataset_dir / "y_train.npy"
    )
    val_ds = EmotionMultimodalDataset(
        dataset_dir / "X_val_images.npy",
        dataset_dir / "X_val_landmarks.npy",
        dataset_dir / "y_val.npy"
    )
    test_ds = EmotionMultimodalDataset(
        dataset_dir / "X_test_images.npy",
        dataset_dir / "X_test_landmarks.npy",
        dataset_dir / "y_test.npy"
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # Print info
    y_train = np.load(dataset_dir / "y_train.npy")
    counts = Counter(y_train.tolist())
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    print(f"Train distribution:")
    for i, emo in enumerate(EMOTIONS):
        print(f"  {emo:>10s}: {counts.get(i, 0)}")

    return train_loader, val_loader, test_loader

# Load original dataset
train_loader, val_loader, test_loader = load_dataloaders(DATASET_DIR, BATCH_SIZE)

## 3. Skenario B1: Baseline (Tanpa Class Weights)

In [ ]:
# B1: Baseline - no class weights
model_b1 = IntermediateFusionTransfer(num_classes=NUM_CLASSES, landmark_dim=136, pretrained=True).to(device)
criterion_b1 = nn.CrossEntropyLoss()
optimizer_b1 = torch.optim.Adam(model_b1.parameters(), lr=LR)
scheduler_b1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b1, mode="max", factor=0.5, patience=8, min_lr=1e-7
)

print(f"Model parameters: {sum(p.numel() for p in model_b1.parameters()):,}")
print("Training B1 (baseline)...")
history_b1, best_epoch_b1 = train_model(
    model_b1, train_loader, val_loader, criterion_b1, optimizer_b1, scheduler_b1,
    device, model_type="fusion", epochs=EPOCHS, patience=PATIENCE,
    save_path=str(OUTPUT_DIR / "intermediate_tl_b1_baseline.pth")
)

In [ ]:
plot_training_history(history_b1, "Intermediate Fusion TL B1 - Baseline")

In [ ]:
# Evaluate B1 on test set
print("=" * 60)
print("EVALUASI B1 - BASELINE")
print("=" * 60)
results_b1 = full_evaluation(model_b1, test_loader, criterion_b1, device, "fusion")
plot_confusion_matrix(results_b1["confusion_matrix"], "Intermediate Fusion TL B1 - Baseline")

## 4. Skenario B2: Dengan Class Weights

In [ ]:
# B2: With class weights (Cui et al., 2019)
weights = get_class_weights(DATASET_DIR, device)
print(f"Class weights: {weights}")

model_b2 = IntermediateFusionTransfer(num_classes=NUM_CLASSES, landmark_dim=136, pretrained=True).to(device)
criterion_b2 = nn.CrossEntropyLoss(weight=weights)
optimizer_b2 = torch.optim.Adam(model_b2.parameters(), lr=LR)
scheduler_b2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b2, mode="max", factor=0.5, patience=8, min_lr=1e-7
)

print("\nTraining B2 (class weights)...")
history_b2, best_epoch_b2 = train_model(
    model_b2, train_loader, val_loader, criterion_b2, optimizer_b2, scheduler_b2,
    device, model_type="fusion", epochs=EPOCHS, patience=PATIENCE,
    save_path=str(OUTPUT_DIR / "intermediate_tl_b2_weighted.pth")
)

In [ ]:
plot_training_history(history_b2, "Intermediate Fusion TL B2 - Class Weights")

print("=" * 60)
print("EVALUASI B2 - CLASS WEIGHTS")
print("=" * 60)
results_b2 = full_evaluation(model_b2, test_loader, criterion_b2, device, "fusion")
plot_confusion_matrix(results_b2["confusion_matrix"], "Intermediate Fusion TL B2 - Class Weights")

## 5. Skenario B3: Class Weights + Augmentasi

In [ ]:
# B3: Class weights + augmented data
train_loader_aug, _, _ = load_dataloaders(DATASET_AUG_DIR, BATCH_SIZE)
weights_aug = get_class_weights(DATASET_AUG_DIR, device)
print(f"Augmented class weights: {weights_aug}")

model_b3 = IntermediateFusionTransfer(num_classes=NUM_CLASSES, landmark_dim=136, pretrained=True).to(device)
criterion_b3 = nn.CrossEntropyLoss(weight=weights_aug)
optimizer_b3 = torch.optim.Adam(model_b3.parameters(), lr=LR)
scheduler_b3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b3, mode="max", factor=0.5, patience=8, min_lr=1e-7
)

print("\nTraining B3 (class weights + augmentation)...")
history_b3, best_epoch_b3 = train_model(
    model_b3, train_loader_aug, val_loader, criterion_b3, optimizer_b3, scheduler_b3,
    device, model_type="fusion", epochs=EPOCHS, patience=PATIENCE,
    save_path=str(OUTPUT_DIR / "intermediate_tl_b3_augmented.pth")
)

In [ ]:
plot_training_history(history_b3, "Intermediate Fusion TL B3 - Class Weights + Augmentasi")

print("=" * 60)
print("EVALUASI B3 - CLASS WEIGHTS + AUGMENTASI")
print("=" * 60)
results_b3 = full_evaluation(model_b3, test_loader, criterion_b3, device, "fusion")
plot_confusion_matrix(results_b3["confusion_matrix"], "Intermediate Fusion TL B3 - Augmented")

## 6. Perbandingan 3 Skenario

In [ ]:
# Perbandingan per-class F1 score
all_results = {
    "B1 Baseline": results_b1,
    "B2 Class Weights": results_b2,
    "B3 Weights+Aug": results_b3,
}
plot_per_class_f1(all_results, "Intermediate Fusion TL - Perbandingan F1 Score per Emosi")

# Summary table
print("=" * 70)
print("RINGKASAN INTERMEDIATE FUSION TRANSFER LEARNING - 3 SKENARIO")
print("=" * 70)
print(f"{'Skenario':<25} {'Accuracy':>10} {'Macro F1':>10} {'Weighted F1':>12}")
print("-" * 70)
for name, r in all_results.items():
    print(f"{name:<25} {r['test_accuracy']:>10.4f} {r['test_macro_f1']:>10.4f} {r['test_weighted_f1']:>12.4f}")

## 7. Simpan Hasil

In [ ]:
# Save results
fusion_results = {}
for name, r in all_results.items():
    fusion_results[name] = {
        "accuracy": float(r["test_accuracy"]),
        "macro_f1": float(r["test_macro_f1"]),
        "weighted_f1": float(r["test_weighted_f1"]),
    }
with open(OUTPUT_DIR / "intermediate_fusion_transfer_results.json", "w") as f:
    json.dump(fusion_results, f, indent=2)
print(f"Results saved to {OUTPUT_DIR / 'intermediate_fusion_transfer_results.json'}")